## Вариант 1

Исходные данные для средних температур по декадам - набор данных `aggregated_data.csv`.

**Цель:**
- Оценить матрицу перехода между температурными диапазонами первой декады февраля и первой декады мая по статистике примерно 10 лет.
- Используя эту матрицу и известную температуру в текущем году в первую декаду февраля, вычислить распределение вероятностей по температурным диапазонам для первой декады мая:
  
  $$ \pi(10) = P^T(1,10) \cdot \pi(1) $$

Где:
- $ \pi(1) $ — начальный вектор распределения (one-hot: мы точно знаем температуру в первую декаду февраля текущего года),
- $ P(1,10) $ — матрица условных вероятностей перехода из диапазона температур в феврале в диапазон температур в мае (оценена по историческим данным),
- $ P^T(1,10) $ — транспонированная матрица $ P(1,10) $, чтобы удобно умножать на столбец $ \pi(1) $.

По условию:
- Используем **единую матрицу** переходов, заполненную по статистике **условных вероятностей перехода между температурными диапазонами только для первых декад февраля и мая**.
- Вся информация о промежуточных декад (март, апрель и др.) **игнорируется**.
- Работаем с диапазоном **март 2016 — май 2025 (и далее)**, но **вместо февраля 2016 используем февраль 2026** (для получения пары «февраль–май» и сохранения длины выборки).

Далее идут шаги подготовки данных, построения матрицы переходов и вычисления $ \pi(10) $.

In [ ]:
import pandas as pd
import numpy as np

csv_path = 'aggregated_data.csv'

df = pd.read_csv(csv_path, sep=';')

df = df.sort_values(['year', 'month', 'decade']).reset_index(drop=True)

df.head()

,year,month,decade,T_avg
0,2016,3,1,-12.56
1,2016,3,2,-8.94
2,2016,3,3,-3.35
3,2016,4,1,-1.79
4,2016,4,2,-4.50


## Выбор периодов и формирование пар (февраль, май)

По условию нам нужны только **первые декады февраля и мая**.

- Диапазон лет для мая: с **2016 по 2025** (включительно) — это даёт нам значения $T_{\text{May},1}$ для 10 лет.
- Для февраля должны быть соответствующие $T_{\text{Feb},1}$ по тем же годам. Однако в данных нет февраля 2016, поэтому по условию **вместо февраля 2016 используем февраль 2026**.

Таким образом, формируем 10 пар:
- (февраль 2026, май 2016),
- (февраль 2017, май 2017),
- ...,
- (февраль 2025, май 2025).

Каждой паре сопоставим пару температурных диапазонов: начальный (февраль) и конечный (май). На основании всех таких пар оценим условные вероятности перехода из диапазона в феврале в диапазон в мае.

In [ ]:
may_first = df[(df['month'] == 5) & (df['decade'] == 1) & (df['year'].between(2016, 2025))]

feb_first_regular = df[(df['month'] == 2) & (df['decade'] == 1) & (df['year'].between(2017, 2025))]

# Отдельно первая декада февраля 2026 (для пары с маем 2016)
feb_2026 = df[(df['year'] == 2026) & (df['month'] == 2) & (df['decade'] == 1)]

may_first, feb_first_regular, feb_2026

(     year  month  decade  T_avg
 6    2016      5       1   7.27
 42   2017      5       1   4.60
 78   2018      5       1   6.05
 114  2019      5       1  12.90
 150  2020      5       1   3.40
 186  2021      5       1   6.85
 222  2022      5       1   3.85
 258  2023      5       1   4.95
 294  2024      5       1   4.80
 330  2025      5       1   5.05,
      year  month  decade  T_avg
 33   2017      2       1 -10.89
 69   2018      2       1 -10.95
 105  2019      2       1 -16.95
 141  2020      2       1 -19.45
 177  2021      2       1 -13.80
 213  2022      2       1 -11.75
 249  2023      2       1 -19.40
 285  2024      2       1 -18.45
 321  2025      2       1 -19.95,
      year  month  decade  T_avg
 357  2026      2       1  -14.5)

In [ ]:
# Формируем единую таблицу пар (T_feb, T_may)

pair_2016 = pd.DataFrame({
    'year_pair': [2016],
    'T_feb': feb_2026['T_avg'].values,
    'T_may': may_first[may_first['year'] == 2016]['T_avg'].values,
})

pairs_regular = []
for y in range(2017, 2026):
    t_feb = feb_first_regular.loc[feb_first_regular['year'] == y, 'T_avg'].values
    t_may = may_first.loc[may_first['year'] == y, 'T_avg'].values
    if len(t_feb) == 1 and len(t_may) == 1:
        pairs_regular.append({'year_pair': y, 'T_feb': t_feb[0], 'T_may': t_may[0]})

pairs_regular = pd.DataFrame(pairs_regular)

pairs = pd.concat([
    pair_2016,
    pairs_regular
], ignore_index=True)

pairs

,year_pair,T_feb,T_may
0,2016,-14.50,7.27
1,2017,-10.89,4.60
2,2018,-10.95,6.05
3,2019,-16.95,12.90
4,2020,-19.45,3.40
5,2021,-13.80,6.85
6,2022,-11.75,3.85
7,2023,-19.40,4.95
8,2024,-18.45,4.80
9,2025,-19.95,5.05


## Дискретизация температуры на диапазоны (состояния цепи Маркова)

Для построения марковской модели нам нужно перейти от непрерывных значений температуры к **конечному числу состояний**.

Зададим \(N\) диапазонов температур (коридоров), не обязательно равных по величине. В этом примере для наглядности выберем:

- **N = 5** диапазонов,
- границы возьмём как **равные по ширине интервалы** между минимальной и максимальной температурой, встречающимися в наших парах (февраль–май).

То есть:
- найдём $T_{\min}$ и $T_{\max}$ по всем значениям `T_feb` и `T_may` из таблицы `pairs`,
- разобьём отрезок $[T_{\min}, T_{\max}]$ на 5 равных интервалов,
- каждое значение температуры отнесём к одному из этих интервалов и присвоим ему **индекс состояния** $k \in \{0,1,2,3,4\}$.

На основе этих индексов построим матрицу переходов $P(1,10)$.

In [ ]:
# Дискретизация температур в парах на N диапазонов

N_STATES = 5

all_temps = np.concatenate([pairs['T_feb'].values, pairs['T_may'].values])
T_min, T_max = all_temps.min(), all_temps.max()

# Небольшой «запас» по краям, чтобы избежать принадлежности к граничным точкам
margin = 1e-6
bin_edges = np.linspace(T_min - margin, T_max + margin, N_STATES + 1)

bin_edges

array([-19.950001 , -13.3800006,  -6.8100002,  -0.2399998,   6.3300006,
        12.900001 ])

In [ ]:
# Функция для перевода температуры в индекс состояния

def temp_to_state(t: float, edges: np.ndarray) -> int:
    """Возвращает индекс интервала [0, N_STATES-1] для температуры t."""
    # pd.cut возвращает категорию; используем right=False, чтобы включать левую границу
    cat = pd.cut([t], bins=edges, right=False, labels=False)
    return int(cat[0])

# Добавляем индексы состояний для февраля и мая
pairs['state_feb'] = pairs['T_feb'].apply(lambda x: temp_to_state(x, bin_edges))
pairs['state_may'] = pairs['T_may'].apply(lambda x: temp_to_state(x, bin_edges))

pairs

,year_pair,T_feb,T_may,state_feb,state_may
0,2016,-14.50,7.27,0,4
1,2017,-10.89,4.60,1,3
2,2018,-10.95,6.05,1,3
3,2019,-16.95,12.90,0,4
4,2020,-19.45,3.40,0,3
5,2021,-13.80,6.85,0,4
6,2022,-11.75,3.85,1,3
7,2023,-19.40,4.95,0,3
8,2024,-18.45,4.80,0,3
9,2025,-19.95,5.05,0,3


## Оценка матрицы переходов $P(1,10)$

Теперь по сформированным парам состояний `(state_feb, state_may)` оценим **условные вероятности перехода**:

$$ P(1,10)_{ij} = \mathbb{P}(\text{состояние в мае} = j \,\mid\, \text{состояние в феврале} = i). $$

Практически:
- для каждого начального состояния `i` посчитаем, сколько раз оно встречается в наших парах,
- для каждого перехода `i → j` посчитаем количество таких наблюдений,
- нормируем по строкам (по `i`), чтобы суммы в строках были равны 1.

Если какая-то строка не встретилась в данных (нет ни одного года с таким диапазоном в феврале),
то для неё введём **равномерное распределение** по всем состояниям в мае (чтобы матрица оставалась стохастической).

In [ ]:
# Строим матрицу переходов P(1,10)

P = np.zeros((N_STATES, N_STATES), dtype=float)

for _, row in pairs.iterrows():
    i = int(row['state_feb'])
    j = int(row['state_may'])
    P[i, j] += 1.0

# Нормировка по строкам (по начальному состоянию)
row_sums = P.sum(axis=1, keepdims=True)

for i in range(N_STATES):
    if row_sums[i] > 0:
        P[i, :] /= row_sums[i]
    else:
        # Если нет наблюдений для этого состояния в феврале — равномерное распределение
        P[i, :] = 1.0 / N_STATES

P_df = pd.DataFrame(P, columns=[f'state_may_{j}' for j in range(N_STATES)])
P_df.index = [f'state_feb_{i}' for i in range(N_STATES)]
P_df

,state_may_0,state_may_1,state_may_2,state_may_3,state_may_4
state_feb_0,0.0,0.0,0.0,0.571429,0.428571
state_feb_1,0.0,0.0,0.0,1.000000,0.000000
state_feb_2,0.2,0.2,0.2,0.200000,0.200000
state_feb_3,0.2,0.2,0.2,0.200000,0.200000
state_feb_4,0.2,0.2,0.2,0.200000,0.200000


## Формирование начального распределения $\pi(1)$ и вычисление $\pi(10)$

По условию начальное распределение $\pi(1)$ — **дегеративное (one-hot)**: мы точно знаем, в каком температурном диапазоне находится средняя температура
в первую декаду февраля текущего года.

В рамках этого ноутбука в качестве «текущего года» возьмём **2026 год** (у нас есть фактическое значение средней температуры в первую декаду февраля 2026 года). Тогда:

1. Берём температуру первой декады февраля 2026 года из данных.
2. Находим её температурный диапазон (индекс состояния `k_current`), используя те же `bin_edges`.
3. Формируем вектор $\pi(1)$ размера `N_STATES`, где `pi1[k_current] = 1`, остальные элементы равны 0.

Матрица $P(1,10)$ у нас хранится как **строчно-сто­хастическая**: строка `i` — распределение состояний в мае при заданном состоянии `i` в феврале.
Чтобы использовать формулу из условия

$$ \pi(10) = P^T(1,10) \cdot \pi(1), $$

будем интерпретировать $\pi(1)$ как **столбец**, а матрицу переходов — как
$P(1,10)$, и умножать **транспонированную** матрицу на $\pi(1)$.

Результат $\pi(10)$ — вектор распределения вероятностей по тем же температурным диапазонам для первой декады мая при известной температуре в феврале 2026 года.

In [ ]:
# Извлекаем фактическую температуру первой декады февраля 2026 года

feb_2026_T = float(feb_2026['T_avg'].iloc[0])
state_current = temp_to_state(feb_2026_T, bin_edges)

# Формируем вектор pi(1) (столбец, но храним как 1D-массив)
pi1 = np.zeros(N_STATES)
pi1[state_current] = 1.0

# Вычисляем pi(10) = P^T * pi(1)
PT = P.T
pi10 = PT @ pi1

pi1_df = pd.DataFrame({'pi1': pi1})
pi10_df = pd.DataFrame({'pi10': pi10})

print('Температура первой декады февраля 2026 года:', feb_2026_T)
print('Индекс состояния (февраль 2026):', state_current)
print('\nНачальное распределение pi(1):')
display(pi1_df.T)

print('\nРаспределение pi(10) для первой декады мая:')
display(pi10_df.T)

Температура первой декады февраля 2026 года: -14.5
Индекс состояния (февраль 2026): 0

Начальное распределение pi(1):


,0,1,2,3,4
pi1,1.0,0.0,0.0,0.0,0.0



Распределение pi(10) для первой декады мая:


,0,1,2,3,4
pi10,0.0,0.0,0.0,0.571429,0.428571


## Оценка температуры первой декады мая 2026 года по распределению $\pi(10)$

Мы получили вектор распределения $\pi(10)$ по температурным диапазонам (состояниям) для первой декады мая при известной температуре в первую декаду февраля 2026 года.

Чтобы получить **численную оценку самой температуры** для первой декады мая 2026 года, интерпретируем это распределение как вероятности попадания температуры в каждый из интервалов и вычислим **математическое ожидание**:

$$
\mathbb{E}[T_{\text{May},2026}] = \sum_{k=0}^{N_{states}-1} \pi_{10,k} \cdot T_k^{(rep)},
$$

где $T_k^{(rep)}$ — представительное значение температуры для интервала $k$. В качестве $T_k^{(rep)}$ возьмём **середину интервала** по границам `bin_edges`.

Таким образом, получим прогностическую оценку средней температуры первой декады мая 2026 года на основе построенной марковской модели.

In [ ]:
# Оценка ожидаемой температуры первой декады мая 2026 года

# Представительные значения температуры для каждого состояния (середины интервалов)
state_repr_temps = (bin_edges[:-1] + bin_edges[1:]) / 2

# Проверим длины на всякий случай
assert len(state_repr_temps) == N_STATES
assert len(pi10) == N_STATES

# Математическое ожидание температуры по распределению pi(10)
expected_T_may_2026 = float(np.dot(pi10, state_repr_temps))

print("Представительные температуры по состояниям (середины интервалов):")
for k, t_mid in enumerate(state_repr_temps):
    print(f"  state {k}: {t_mid:.2f} °C")

print("\nРаспределение pi(10) по состояниям:")
for k, p in enumerate(pi10):
    print(f"  state {k}: {p:.4f}")

print(f"\nОценка средней температуры первой декады мая 2026 года по модели: {expected_T_may_2026:.2f} °C")

Представительные температуры по состояниям (середины интервалов):
  state 0: -16.67 °C
  state 1: -10.10 °C
  state 2: -3.52 °C
  state 3: 3.05 °C
  state 4: 9.62 °C

Распределение pi(10) по состояниям:
  state 0: 0.0000
  state 1: 0.0000
  state 2: 0.0000
  state 3: 0.5714
  state 4: 0.4286

Оценка средней температуры первой декады мая 2026 года по модели: 5.86 °C
